## Cell 1

In [ ]:
!pip install -q nbformat gradio

## Cell 2

In [ ]:
import os
import re
import nbformat
import gradio as gr

from pathlib import Path

## Cell 3

In [ ]:
def convert_markdown_to_notebook(file_path):

    if file_path is None:
        raise gr.Error("Please upload a Markdown file.")

    file_path = Path(file_path)

    if file_path.suffix.lower() != ".md":
        raise gr.Error("Please upload a .md file.")

    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()

    notebook = nbformat.v4.new_notebook()

    cells = []
    markdown_buffer = []
    code_buffer = []

    inside_code = False
    code_language = ""

    lines = content.splitlines()

    for line in lines:

        fence = re.match(r"^\s*(`{3,}|~{3,})(.*)$", line)

        if fence:

            if not inside_code:

                if markdown_buffer:
                    markdown_text = "\n".join(markdown_buffer).strip()

                    if markdown_text:
                        cells.append(
                            nbformat.v4.new_markdown_cell(markdown_text)
                        )

                    markdown_buffer = []

                inside_code = True
                fence_character = fence.group(1)[0]
                fence_length = len(fence.group(1))
                code_language = fence.group(2).strip().split()[0] if fence.group(2).strip() else ""

            elif fence.group(1)[0] == fence_character and len(fence.group(1)) >= fence_length:

                code_text = "\n".join(code_buffer)

                code_cell = nbformat.v4.new_code_cell(code_text)

                if code_language:
                    code_cell.metadata["language"] = code_language

                cells.append(code_cell)

                code_buffer = []
                inside_code = False
                code_language = ""

            continue

        if inside_code:
            code_buffer.append(line)
        else:
            markdown_buffer.append(line)

    if inside_code:
        raise gr.Error("An unclosed code block was found in the Markdown file.")

    if markdown_buffer:
        markdown_text = "\n".join(markdown_buffer).strip()

        if markdown_text:
            cells.append(
                nbformat.v4.new_markdown_cell(markdown_text)
            )

    notebook.cells = cells

    notebook.metadata = {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python"
        }
    }

    output_path = file_path.with_name(
        file_path.stem + ".ipynb"
    )

    nbformat.validate(notebook)

    nbformat.write(notebook, output_path)

    return str(output_path)

## Cell 4

In [ ]:
with gr.Blocks(title="Markdown to Notebook Converter") as app:

    gr.Markdown(
        """
        # Markdown to Jupyter Notebook Converter

        Upload a Markdown file and convert it into
        a downloadable Jupyter Notebook.

        - Markdown text → Markdown cells
        - Python code blocks → Code cells
        """
    )

    input_file = gr.File(
        label="Upload Markdown File",
        type="filepath"
    )

    convert_button = gr.Button(
        "Convert to Notebook",
        variant="primary"
    )

    output_file = gr.File(
        label="Download Jupyter Notebook"
    )

    convert_button.click(
        fn=convert_markdown_to_notebook,
        inputs=input_file,
        outputs=output_file
    )

app.launch(share=True)